# Telco Customer Churn - Projet Data Science orienté business

**Objectif:** prédire les clients à risque de résiliation (churn) pour aider une équipe marketing/rétention à agir avant le départ client.

**Positionnement recruteur:** ce notebook montre un workflow complet: compréhension métier, préparation des données, modélisation, évaluation et interprétation.

## 1. Contexte métier

Dans le secteur télécom, conserver un client coûte souvent moins cher qu'en acquérir un nouveau.
Ce projet répond à une question opérationnelle simple:

> Quels clients ont le plus de probabilité de quitter le service, et comment prioriser les actions de rétention ?

## 2. Import des librairies

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

## 3. Chargement des données

Le notebook cherche le fichier CSV dans le dossier courant, puis dans `~/Téléchargements`.

In [ ]:
DATA_FILENAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

candidate_paths = [
    Path(DATA_FILENAME),
    Path.home() / "Téléchargements" / DATA_FILENAME,
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        f"Fichier introuvable: {DATA_FILENAME}. Place-le dans le dossier du notebook ou dans ~/Téléchargements."
    )

df = pd.read_csv(data_path)
print(f"Fichier chargé: {data_path}")
print(f"Dimensions: {df.shape}")
df.head()

## 4. Contrôle qualité des données

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
duplicates_before = df.duplicated().sum()
print(f"Doublons avant nettoyage: {duplicates_before}")

## 5. Nettoyage et prétraitement

Actions réalisées:
- suppression de `customerID` (identifiant non prédictif);
- conversion de `TotalCharges` en numérique;
- suppression des doublons;
- suppression des lignes avec valeurs manquantes résiduelles;
- encodage binaire de la cible `Churn`.

In [ ]:
df = df.drop(columns=["customerID"], errors="ignore").copy()

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df = df.drop_duplicates().copy()

df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

df = df.dropna().copy()

print(f"Dimensions après nettoyage: {df.shape}")
print("
Répartition de la cible (0=No churn, 1=Churn):")
print(df["Churn"].value_counts(normalize=True).round(3))

## 6. Analyse exploratoire (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Répartition Churn")
axes[0].set_xlabel("Churn")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("MonthlyCharges selon Churn")
axes[1].set_xlabel("Churn")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, kde=True, multiple="stack")
plt.title("Ancienneté client (tenure) et churn")
plt.xlabel("tenure (mois)")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Type de contrat et churn")
plt.xticks(rotation=15)
plt.show()

## 7. Feature engineering

Encodage one-hot des variables catégorielles (`drop_first=True`) puis séparation `X / y`.

In [ ]:
X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True)
y = df["Churn"]

print(f"Nombre de variables explicatives: {X.shape[1]}")
X.head()

## 8. Séparation train/test

Utilisation d'une séparation stratifiée pour conserver la proportion de churn.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

## 9. Modèle 1 - SVM (avec normalisation)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(probability=True)

param_grid = {
    "C": [0.5, 1, 3],
    "kernel": ["rbf", "linear"],
    "class_weight": [None, "balanced"],
    "random_state": [42],
}

grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)

grid.fit(X_train_scaled, y_train)

best_svm = grid.best_estimator_
print("Meilleurs paramètres SVM:", grid.best_params_)

In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_proba):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

results = []

svm_pred = best_svm.predict(X_test_scaled)
svm_proba = best_svm.predict_proba(X_test_scaled)[:, 1]

results.append(evaluate_model("SVM", y_test, svm_pred, svm_proba))

print(classification_report(y_test, svm_pred))
ConfusionMatrixDisplay.from_predictions(y_test, svm_pred)
plt.title("Matrice de confusion - SVM")
plt.show()

RocCurveDisplay.from_predictions(y_test, svm_proba)
plt.title("ROC - SVM")
plt.show()

## 10. Modèle 2 - XGBoost (si disponible)

Cette section est optionnelle: si `xgboost` n'est pas installé, le notebook continue sans erreur.

In [ ]:
xgb_available = True
try:
    from xgboost import XGBClassifier
except Exception as e:
    xgb_available = False
    print("XGBoost non disponible:", e)

if xgb_available:
    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
    )
    xgb_model.fit(X_train, y_train)

    xgb_pred = xgb_model.predict(X_test)
    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

    results.append(evaluate_model("XGBoost", y_test, xgb_pred, xgb_proba))

    print(classification_report(y_test, xgb_pred))
    ConfusionMatrixDisplay.from_predictions(y_test, xgb_pred)
    plt.title("Matrice de confusion - XGBoost")
    plt.show()

    RocCurveDisplay.from_predictions(y_test, xgb_proba)
    plt.title("ROC - XGBoost")
    plt.show()

## 11. Comparaison des performances

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False).reset_index(drop=True)
results_df.style.format({
    "accuracy": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
})

## 12. Interprétabilité (SHAP, optionnel)

Si `shap` et `xgboost` sont disponibles, on visualise les variables les plus influentes.

In [ ]:
if xgb_available:
    try:
        import shap

        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(X_test)
        shap.summary_plot(shap_values, X_test)
    except Exception as e:
        print("SHAP non disponible ou erreur d'exécution:", e)

## 13. Lecture business des résultats

Exemples d'actions exploitables:

- cibler en priorité les clients à forte probabilité de churn;
- proposer des offres de rétention adaptées au profil contrat/facturation;
- mettre en place un scoring hebdomadaire pour les équipes CRM.


## 14. Conclusion

Ce notebook présente une démarche de bout en bout, de la donnée brute à la recommandation métier.

Pour un portfolio, tu peux ajouter ici:
- ton lien GitHub;
- ton LinkedIn;
- une phrase de valeur: ce que tu peux apporter à une équipe Data/IA.